# Qwen3-ASR Colab ノートブック

このノートブックは、Qwen3-ASRを使用して音声ファイルまたは動画ファイルを文字起こしします。

## 特長

- **無料:** このノートブックは、オープンソースの [Qwen3-ASR](https://github.com/QwenLM/Qwen3-ASR) と Google Colab の無料GPUランタイムを使用します。
- **タイムスタンプ・字幕対応:** `Qwen3-ForcedAligner-0.6B` を使用して単語・形態素単位の時刻情報を取得し、SRT字幕ファイルを生成できます。
- **ノートブック側の固定ファイルサイズ制限なし:** 長い音声・動画ファイルも処理できます。
- **バッチ処理:** 複数の音声・動画ファイルを1回の実行で処理できます。
- **Google Drive対応:** ファイルを直接アップロードするほか、Google Drive内の音声・動画ファイルを処理することもできます。
- **完全オンライン:** モデルのダウンロードと文字起こし処理は Google Colab 上で行われます。ローカルPCの計算資源は使用しません。
- **複数の出力形式:** JSON形式の文字起こし結果に加えて、TXT、CSV、Markdown、Excel、SRT字幕ファイルの形式を出力できます。
- **コンテキスト指定:** `CONTEXT_INFO` に人名、組織名、専門用語などの参考情報を指定できます。
- **多言語対応:** Qwen3-ASRは30言語に対応しています。中国語・広東語は合計22の方言に対応しています。

## 使い方

1. **1. 設定** で `INPUT_MODE` を選びます。必要に応じてモデル、言語、タイムスタンプ、出力設定、その他事項も変更します。
2. **Runtime -> Run all** を選びます。
3. `INPUT_MODE = "upload"` の場合は、**2. ファイル選択 / Google Driveマウント** が始まるまで待ち、セルの出力欄に表示される **Choose Files** をクリックしてファイルをアップロードします。
4. 処理が完了すると、`AUTO_DOWNLOAD_ZIP = True` の場合は文字起こし結果をまとめたZIPファイルが自動的にダウンロードされます。`SAVE_OUTPUTS_TO_GOOGLE_DRIVE = True` の場合は、Google DriveにもZIPファイルが保存されます。

## 備考

- Google Colab の無料 GPU空き状況には限りがあります。利用可能な GPUがなく、`REQUIRE_CUDA = True`の場合、このノートブックは文字起こし前に停止します。
- 処理には時間がかかります。 目安として、60 分の音声またはビデオ ファイルの文字起こしには10分程度かかることがあります。初回実行時は、依存ライブラリとQwen3-ASRモデルをダウンロードするため、文字起こし開始まで時間がかかる場合があります。ファイルを直接アップロードする場合は、アップロード時間も必要です。
- 処理中はこのノートブックを開いたままにしてください。文字起こし中にランタイムが切断されると、処理は停止します。


In [ ]:
# @title 1. 設定
# ============================================================
# 1. 設定
# ============================================================
from pathlib import Path

# @markdown ## 入力
# @markdown - `INPUT_MODE`: ローカル ファイルには `upload`を、Google Driveに既に保存されているファイルには `google_drive`を使用します。
INPUT_MODE = "upload"  # @param ["upload", "google_drive"]

# @markdown ### Google Drive入力
# @markdown これらの設定は、`INPUT_MODE == "google_drive"`の場合にのみ使用されます。
# @markdown - `GOOGLE_DRIVE_ROOT`: 通常、これは `/content/drive/MyDrive`に設定しておいてください。
# @markdown - `GOOGLE_DRIVE_SUBPATH`: MyDrive 下のファイルまたはフォルダのパス。最上位の MyDrive フォルダを使用する場合は、空白のままにします。
# @markdown - `RECURSIVE`: ドライブの入力パスがフォルダの場合は、サブフォルダも検索します。
# @markdown - `FILENAME_CONTAINS`: オプションのファイル名の部分文字列フィルタ。
GOOGLE_DRIVE_ROOT = "/content/drive/MyDrive"  # @param {type:"string"}
GOOGLE_DRIVE_SUBPATH = ""  # @param {type:"string"}
RECURSIVE = False  # @param {type:"boolean"}
FILENAME_CONTAINS = ""  # @param {type:"string"}

# @markdown ## Qwen3-ASRモデル
# @markdown - `MODEL_SIZE`: "1.7B": 品質優先 / "0.6B": 軽量・速度優先
# @markdown - `SOURCE_LANGUAGE`: 不明な場合は `Auto`を指定できます
# @markdown - `CONTEXT_INFO`: 人名、組織名、専門用語などの参考情報を指定できます
MODEL_SIZE = "1.7B"  # @param ["1.7B", "0.6B"]
SOURCE_LANGUAGE = "Japanese"  # @param ["Auto", "Chinese", "English", "Cantonese", "Arabic", "German", "French", "Spanish", "Portuguese", "Indonesian", "Italian", "Korean", "Russian", "Thai", "Vietnamese", "Japanese", "Turkish", "Hindi", "Malay", "Dutch", "Swedish", "Danish", "Finnish", "Polish", "Czech", "Filipino", "Persian", "Greek", "Hungarian", "Macedonian", "Romanian"]
CONTEXT_INFO = ""  # @param {type:"string"}

# @markdown ## タイムスタンプ
# @markdown - `none`: 時刻を付けない
# @markdown - `word`: 単語/形態素レベルの時刻をJSON/CSV/XLSXへ保存
# @markdown - `subtitle`: wordに加えてSRT字幕ファイルを生成
TIMESTAMP_MODE = "subtitle"  # @param ["subtitle", "word", "none"]

# 字幕（内部の約180秒ForcedAligner chunkとは独立）
SUBTITLE_MAX_SECONDS = 6.0
SUBTITLE_MAX_CHARS = 36
SUBTITLE_GAP_BREAK_SECONDS = 0.8
SUBTITLE_MIN_SENTENCE_BREAK_SECONDS = 0.8
SUBTITLE_MAX_LINE_CHARS = 20

# @markdown ## 出力
# @markdown - `SAVE_OUTPUTS_TO_GOOGLE_DRIVE`: `True` の場合、入力方法にかかわらず出力ZIPを Google Driveに保存します。
# @markdown - `AUTO_DOWNLOAD_ZIP`: 最後に ZIPアーカイブを 1 つ作成してダウンロードします。
# - 出力形式は以下のブール値によって制御されます。
SAVE_OUTPUTS_TO_GOOGLE_DRIVE = False  # @param {type:"boolean"}
AUTO_DOWNLOAD_ZIP = True  # @param {type:"boolean"}
SAVE_TXT = True
SAVE_MD = True
SAVE_JSON = True
SAVE_CSV = True
SAVE_XLSX = True
SAVE_SRT = True
INCLUDE_TIMESTAMPS_IN_TXT = True

# Advanced
REQUIRE_CUDA = True
MAX_INFERENCE_BATCH_SIZE = 1
MAX_NEW_TOKENS = 2048
UPLOAD_DIR = "/content/qwen3_asr_uploads"
WORK_DIR = "/content/qwen3_asr_work"
OUTPUT_DIR = "/content/qwen3_asr_outputs"
ZIP_FILE_NAME = "qwen3_asr_outputs.zip"

SUPPORTED_EXTS = {
    ".aac", ".avi", ".flac", ".m4a", ".m4v", ".mkv", ".mov", ".mp3",
    ".mp4", ".ogg", ".opus", ".wav", ".webm", ".wma",
}

_drive_root = str(Path(GOOGLE_DRIVE_ROOT.strip().rstrip("/")))
_drive_subpath = GOOGLE_DRIVE_SUBPATH.strip().strip("/")
DRIVE_INPUT_PATH = str(Path(_drive_root) / _drive_subpath) if _drive_subpath else _drive_root

MODEL_ID = f"Qwen/Qwen3-ASR-{MODEL_SIZE}"
ALIGNER_ID = "Qwen/Qwen3-ForcedAligner-0.6B"
RETURN_TIMESTAMPS = TIMESTAMP_MODE in {"word", "subtitle"}
FORCED_ALIGNER_LANGUAGES = {"Cantonese", "Chinese", "English", "French", "German", "Italian", "Japanese", "Korean", "Portuguese", "Russian", "Spanish"}
if (
    SOURCE_LANGUAGE != "Auto"
    and RETURN_TIMESTAMPS
    and SOURCE_LANGUAGE not in FORCED_ALIGNER_LANGUAGES
):
    raise ValueError(
        "Qwen3-ForcedAligner-0.6B は {language} のタイムスタンプ生成に対応していません。`TIMESTAMP_MODE` を `none` にするか、`SOURCE_LANGUAGE` を `Auto` または ForcedAligner 対応言語に変更してください。".format(language=SOURCE_LANGUAGE)
    )

print("設定を読み込みました。")
print("MODEL_ID =", MODEL_ID)
print("SOURCE_LANGUAGE =", SOURCE_LANGUAGE)
print("TIMESTAMP_MODE =", TIMESTAMP_MODE)


In [ ]:
# @title 2. ファイル選択 / Google Driveマウント
# ============================================================
# 2. ファイル選択 / Google Driveマウント
# ============================================================
from pathlib import Path
import shutil

Path(UPLOAD_DIR).mkdir(parents=True, exist_ok=True)
Path(WORK_DIR).mkdir(parents=True, exist_ok=True)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

def is_supported_media(path: Path) -> bool:
    return path.is_file() and path.suffix.lower() in SUPPORTED_EXTS

def collect_drive_files(path_str: str) -> list[str]:
    root = Path(path_str)
    if root.is_file():
        if not is_supported_media(root):
            raise ValueError(f"対応していないファイル形式です: {root}")
        return [str(root)]
    if not root.exists():
        raise FileNotFoundError(f"Google Driveのパスが存在しません: {root}")
    iterator = root.rglob("*") if RECURSIVE else root.glob("*")
    files_found = []
    for p in iterator:
        if is_supported_media(p):
            if FILENAME_CONTAINS and FILENAME_CONTAINS not in p.name:
                continue
            files_found.append(str(p))
    return sorted(files_found)

input_files = []

if INPUT_MODE == "upload":
    from google.colab import files
    print("音声・動画ファイルを選択してください。複数選択できます。")
    uploaded = files.upload()
    for name in uploaded:
        src = Path(name)
        dst = Path(UPLOAD_DIR) / src.name
        if src.exists():
            shutil.move(str(src), str(dst))
        if is_supported_media(dst):
            input_files.append(str(dst))
elif INPUT_MODE == "google_drive":
    from google.colab import drive
    drive.mount("/content/drive")
    input_files = collect_drive_files(DRIVE_INPUT_PATH)
else:
    raise ValueError("INPUT_MODE は upload または google_drive を指定してください。")

if not input_files:
    raise RuntimeError("処理対象の音声・動画ファイルがありません。")

print("選択されたファイル:")
for f in input_files:
    print("-", f)


In [ ]:
# @title 3. GPUのチェックと依存関係のインストール
# ============================================================
# 3. GPUのチェックと依存関係のインストール
# ============================================================
import os, shutil, subprocess, sys
from pathlib import Path

os.environ.setdefault("PIP_DISABLE_PIP_VERSION_CHECK", "1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

def run_cmd(cmd):
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)

if not shutil.which("ffmpeg"):
    run_cmd(["apt-get", "update", "-qq"])
    run_cmd(["apt-get", "install", "-y", "ffmpeg"])

flag = Path("/content/qwen3_asr_dependencies_v006.flag")
if not flag.exists():
    run_cmd([
        sys.executable, "-m", "pip", "install", "-q",
        "qwen-asr==0.0.6", "pandas", "openpyxl"
    ])
    flag.write_text("installed", encoding="utf-8")

import torch
print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if REQUIRE_CUDA and not torch.cuda.is_available():
    raise RuntimeError("Google ColabのランタイムをGPUに変更してください。")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {props.total_memory / 1024**3:.2f} GiB")
    print("BF16対応:", torch.cuda.is_bf16_supported())


In [ ]:
# @title 4. ヘルパー関数
# ============================================================
# 4. ヘルパー関数
# ============================================================
from pathlib import Path
import csv, difflib, json, re, shutil, subprocess, unicodedata, zipfile
import pandas as pd
from typing import Any

_SENTENCE_END_RE = re.compile(r'[。！？!?…](?:["”’）】』」]*)\s*$')

def run_ffmpeg(command):
    cp = subprocess.run(command, capture_output=True, text=True, check=False)
    if cp.returncode != 0:
        raise RuntimeError((cp.stderr or cp.stdout or "ffmpegの実行に失敗しました").strip())

def extract_audio(src_path: str) -> Path:
    src = Path(src_path)
    out_dir = Path(WORK_DIR) / "audio"
    out_dir.mkdir(parents=True, exist_ok=True)
    out = out_dir / f"{src.stem}.qwen3.wav"
    run_ffmpeg([
        "ffmpeg", "-y", "-hide_banner", "-loglevel", "error", "-nostdin",
        "-i", str(src), "-map", "0:a:0", "-vn", "-ac", "1", "-ar", "16000",
        "-c:a", "pcm_s16le", str(out)
    ])
    if not out.exists() or out.stat().st_size < 1000:
        raise RuntimeError(f"音声の抽出に失敗しました: {src}")
    return out

def _item_dict(item: Any) -> dict:
    if isinstance(item, dict):
        return {
            "text": str(item.get("text", "")),
            "start_time": float(item.get("start_time", 0.0)),
            "end_time": float(item.get("end_time", 0.0)),
        }
    return {
        "text": str(getattr(item, "text", "")),
        "start_time": float(getattr(item, "start_time", 0.0)),
        "end_time": float(getattr(item, "end_time", 0.0)),
    }


def alignment_to_rows(alignment: Any) -> list[dict]:
    if alignment is None:
        return []
    items = getattr(alignment, "items", alignment)
    return [_item_dict(x) for x in items]



def repair_alignment_timestamps(
    rows: list[dict],
    *,
    max_plausible_chars_per_second: float = 18.0,
) -> tuple[list[dict], dict]:
    """Repair rare non-monotonic ForcedAligner timestamp runs.

    Text/token order remains authoritative. When a timestamp inversion is found,
    the function first tests whether the preceding overshoot run can plausibly
    fit between the last stable time and the current token. If so, only that
    preceding run is redistributed proportionally by visible token length.
    Otherwise the following rollback run is redistributed forward until the
    next stable timestamp. No transcript text is changed.
    """
    repaired = [dict(row) for row in rows]
    inversion_count_before = sum(
        1 for i in range(1, len(repaired))
        if float(repaired[i].get("start_time", 0.0)) < float(repaired[i - 1].get("start_time", 0.0)) - 1e-9
    )
    repairs = []

    for row in repaired:
        start = float(row.get("start_time", 0.0))
        end = float(row.get("end_time", start))
        row["start_time"] = start
        row["end_time"] = max(start, end)

    def token_weight(index: int) -> int:
        text = str(repaired[index].get("text", ""))
        return max(1, sum(1 for ch in text if not ch.isspace()))

    def redistribute(indices: list[int], left: float, right: float) -> None:
        if not indices:
            return
        width = max(0.0, float(right) - float(left))
        weights = [token_weight(i) for i in indices]
        total = max(1, sum(weights))
        cumulative = 0
        for index, weight in zip(indices, weights):
            start = float(left) + width * cumulative / total
            cumulative += weight
            end = float(left) + width * cumulative / total
            repaired[index]["start_time"] = round(start, 3)
            repaired[index]["end_time"] = round(max(start, end), 3)

    i = 1
    safety = 0
    while i < len(repaired):
        if repaired[i]["start_time"] >= repaired[i - 1]["start_time"] - 1e-9:
            i += 1
            continue
        safety += 1
        if safety > 100:
            break

        current_start = repaired[i]["start_time"]
        left_index = i - 1
        while left_index >= 0 and repaired[left_index]["end_time"] > current_start:
            left_index -= 1
        left_end = repaired[left_index]["end_time"] if left_index >= 0 else 0.0
        previous_indices = list(range(left_index + 1, i))
        available = max(0.0, current_start - left_end)
        chars = sum(token_weight(k) for k in previous_indices)
        required_rate = chars / available if available > 1e-6 else float("inf")

        if previous_indices and available > 0.05 and required_rate <= max_plausible_chars_per_second:
            original_start = repaired[previous_indices[0]]["start_time"]
            original_end = repaired[previous_indices[-1]]["end_time"]
            redistribute(previous_indices, left_end, current_start)
            repairs.append({
                "mode": "repair_preceding_overshoot",
                "first_item": previous_indices[0],
                "last_item": previous_indices[-1],
                "original_start": original_start,
                "original_end": original_end,
                "repaired_start": left_end,
                "repaired_end": current_start,
                "required_chars_per_second": required_rate,
            })
            i = max(1, left_index + 1)
            continue

        # The preceding run would have to be implausibly compressed, so treat
        # the current/following timestamps as the rollback side instead.
        target_start = repaired[i - 1]["end_time"]
        next_index = i
        while next_index < len(repaired) and repaired[next_index]["start_time"] < target_start:
            next_index += 1
        following_indices = list(range(i, next_index))
        if next_index < len(repaired):
            target_end = max(target_start, repaired[next_index]["start_time"])
        else:
            target_end = target_start + max(0.05, 0.05 * len(following_indices))
        if target_end <= target_start and next_index < len(repaired):
            target_end = max(target_start, repaired[next_index]["end_time"])

        if following_indices:
            original_start = repaired[following_indices[0]]["start_time"]
            original_end = repaired[following_indices[-1]]["end_time"]
            redistribute(following_indices, target_start, target_end)
            repairs.append({
                "mode": "repair_following_rollback",
                "first_item": following_indices[0],
                "last_item": following_indices[-1],
                "original_start": original_start,
                "original_end": original_end,
                "repaired_start": target_start,
                "repaired_end": target_end,
                "required_chars_per_second": required_rate,
            })
        else:
            repaired[i]["start_time"] = target_start
            repaired[i]["end_time"] = max(target_start, repaired[i]["end_time"])
        i = max(1, i - 1)

    # Defensive final pass: starts must never go backwards and end >= start.
    previous_start = 0.0
    for row in repaired:
        if row["start_time"] < previous_start:
            delta = previous_start - row["start_time"]
            row["start_time"] = previous_start
            row["end_time"] = max(previous_start, row["end_time"] + delta)
        row["end_time"] = max(row["start_time"], row["end_time"])
        previous_start = row["start_time"]

    inversion_count_after = sum(
        1 for i in range(1, len(repaired))
        if repaired[i]["start_time"] < repaired[i - 1]["start_time"] - 1e-9
    )
    diagnostics = {
        "timestamp_inversions_before": inversion_count_before,
        "timestamp_inversions_after": inversion_count_after,
        "timestamp_repair_count": len(repairs),
        "timestamp_repairs": repairs,
        "timestamp_mapping_monotonic": inversion_count_after == 0,
    }
    return repaired, diagnostics

def _is_alignment_char(ch: str) -> bool:
    # Mirrors the relevant part of Qwen3 ForcedAligner's clean_token():
    # keep letters/numbers plus apostrophe; punctuation is ignored for matching.
    if ch == "'":
        return True
    category = unicodedata.category(ch)
    return category.startswith("L") or category.startswith("N")


def _normalize_source(text: str) -> tuple[str, list[int]]:
    chars, raw_indices = [], []
    for raw_i, ch in enumerate(text):
        if _is_alignment_char(ch):
            chars.append(ch)
            raw_indices.append(raw_i)
    return "".join(chars), raw_indices


def _normalize_alignment(rows: list[dict]) -> tuple[str, list[int]]:
    chars, token_owners = [], []
    for token_i, row in enumerate(rows):
        for ch in str(row.get("text", "")):
            if _is_alignment_char(ch):
                chars.append(ch)
                token_owners.append(token_i)
    return "".join(chars), token_owners


def attach_source_fragments(
    source_text: str,
    rows: list[dict],
    *,
    min_similarity: float = 0.995,
) -> tuple[list[dict], dict]:
    """Map the original ASR text back onto ForcedAligner tokens robustly.

    Unlike v1, this does not require every aligner token to occur literally in
    the punctuated source text. It aligns punctuation-stripped Unicode
    letter/number streams globally, then uses matching characters as monotonic
    anchors. Unmatched local tokens are assigned zero-width display fragments,
    allowing the next anchored token to carry the source text without losing
    punctuation or changing the transcript.
    """
    diagnostics = {
        "alignment_mapping_method": "normalized_sequence_alignment",
        "alignment_mapping_min_similarity": float(min_similarity),
    }
    if not rows:
        diagnostics.update({
            "alignment_similarity": 1.0,
            "alignment_normalized_exact": True,
            "alignment_unmatched_source_chars": 0,
            "alignment_unmatched_aligner_chars": 0,
            "alignment_mapping_accepted": True,
            "punctuation_mapping_exact": True,
        })
        return [], diagnostics

    source_norm, source_raw_indices = _normalize_source(source_text)
    align_norm, align_owners = _normalize_alignment(rows)
    matcher = difflib.SequenceMatcher(None, source_norm, align_norm, autojunk=False)
    opcodes = matcher.get_opcodes()
    similarity = float(matcher.ratio())
    unmatched_source = sum(i2 - i1 for tag, i1, i2, j1, j2 in opcodes if tag in {"replace", "delete"})
    unmatched_aligner = sum(j2 - j1 for tag, i1, i2, j1, j2 in opcodes if tag in {"replace", "insert"})

    diagnostics.update({
        "alignment_similarity": similarity,
        "alignment_normalized_exact": source_norm == align_norm,
        "alignment_unmatched_source_chars": int(unmatched_source),
        "alignment_unmatched_aligner_chars": int(unmatched_aligner),
    })

    if not source_norm or not align_norm or similarity < min_similarity:
        fallback = [{**row, "display_text": str(row.get("text", ""))} for row in rows]
        diagnostics.update({
            "alignment_mapping_accepted": False,
            "punctuation_mapping_exact": False,
        })
        return fallback, diagnostics

    # aligner-normalized-char index -> source-normalized-char index
    align_to_source = [None] * len(align_norm)
    for tag, i1, i2, j1, j2 in opcodes:
        if tag != "equal":
            continue
        for offset in range(i2 - i1):
            align_to_source[j1 + offset] = i1 + offset

    # Determine a raw source start anchor for each ForcedAligner token.
    starts = [None] * len(rows)
    for align_char_i, source_norm_i in enumerate(align_to_source):
        if source_norm_i is None:
            continue
        token_i = align_owners[align_char_i]
        raw_pos = source_raw_indices[source_norm_i]
        if starts[token_i] is None or raw_pos < starts[token_i]:
            starts[token_i] = raw_pos

    # Missing local tokens (e.g. a one-character transposition) are collapsed
    # to a zero-width display fragment at the next stable anchor. This keeps
    # timing rows intact without inventing or reordering transcript text.
    next_known = [None] * len(rows)
    nxt = None
    for i in range(len(rows) - 1, -1, -1):
        if starts[i] is not None:
            nxt = starts[i]
        next_known[i] = nxt

    previous = 0
    for i in range(len(rows)):
        if starts[i] is None:
            candidate = next_known[i]
            starts[i] = previous if candidate is None else max(previous, candidate)
        starts[i] = max(previous, min(len(source_text), int(starts[i])))
        previous = starts[i]

    mapped = []
    for i, row in enumerate(rows):
        left = 0 if i == 0 else starts[i]
        right = starts[i + 1] if i + 1 < len(rows) else len(source_text)
        right = max(left, right)
        mapped.append({**row, "display_text": source_text[left:right]})

    reconstructed = "".join(x["display_text"] for x in mapped)
    diagnostics.update({
        "alignment_mapping_accepted": True,
        "punctuation_mapping_exact": reconstructed == source_text,
    })
    return mapped, diagnostics


def _visible_len(text: str) -> int:
    return len(re.sub(r"\\s+", "", text))


def wrap_cue_text(text: str, max_line_chars: int = 20) -> str:
    text = text.strip()
    if max_line_chars <= 0 or _visible_len(text) <= max_line_chars:
        return text
    midpoint = len(text) // 2
    candidates = [i + 1 for i, ch in enumerate(text) if ch.isspace() or ch in "、。！？!?，,；;：:"]
    split_at = min(candidates, key=lambda p: abs(p - midpoint)) if candidates else midpoint
    left, right = text[:split_at].strip(), text[split_at:].strip()
    return left + "\n" + right if left and right else text


def build_subtitle_cues(
    source_text: str,
    alignment_rows: list[dict],
    *,
    max_seconds: float = 6.0,
    max_chars: int = 36,
    gap_break_seconds: float = 0.8,
    min_sentence_break_seconds: float = 0.8,
    max_line_chars: int = 20,
) -> tuple[list[dict], dict, list[dict]]:
    time_rows, timestamp_diag = repair_alignment_timestamps(alignment_rows)
    rows, diagnostics = attach_source_fragments(source_text, time_rows)
    diagnostics.update(timestamp_diag)
    cues, buf, cue_start = [], [], None

    def flush():
        nonlocal buf, cue_start
        if not buf:
            return
        raw_text = "".join(x.get("display_text", x["text"]) for x in buf)
        if raw_text.strip():
            cues.append({
                "start": float(cue_start),
                "end": float(buf[-1]["end_time"]),
                "text": raw_text,
                "display_text": wrap_cue_text(raw_text, max_line_chars=max_line_chars),
            })
        buf, cue_start = [], None

    for i, row in enumerate(rows):
        if cue_start is None:
            cue_start = float(row["start_time"])
        buf.append(row)
        joined = "".join(x.get("display_text", x["text"]) for x in buf)
        duration = float(row["end_time"]) - float(cue_start)
        next_gap = None
        if i + 1 < len(rows):
            next_gap = max(0.0, float(rows[i + 1]["start_time"]) - float(row["end_time"]))
        sentence_end = bool(_SENTENCE_END_RE.search(joined))
        hard_limit = duration >= max_seconds or _visible_len(joined) >= max_chars
        silence_break = next_gap is not None and next_gap >= gap_break_seconds
        natural_break = sentence_end and duration >= min_sentence_break_seconds
        should_flush = hard_limit or silence_break or natural_break
        if should_flush and joined and joined[-1].isspace() and i + 1 < len(rows):
            should_flush = False
        if should_flush:
            flush()
    flush()
    diagnostics["cue_text_exact"] = "".join(c["text"] for c in cues) == source_text
    return cues, diagnostics, rows


def fmt_ts(seconds, srt=False):
    ms = max(0, int(round(float(seconds) * 1000)))
    h, rem = divmod(ms, 3_600_000)
    m, rem = divmod(rem, 60_000)
    s, ms = divmod(rem, 1000)
    sep = "," if srt else "."
    return f"{h:02d}:{m:02d}:{s:02d}{sep}{ms:03d}"

def cues_to_srt(cues):
    blocks = []
    for i, cue in enumerate(cues, 1):
        start = float(cue["start"])
        end = max(start + 0.05, float(cue["end"]))
        blocks.append(
            f"{i}\n{fmt_ts(start, True)} --> {fmt_ts(end, True)}\n{cue.get('display_text', cue['text'])}\n"
        )
    return "\n".join(blocks)

def save_result(src, model_id, result):
    out_dir = Path(OUTPUT_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)
    base = Path(src).stem

    rows = alignment_to_rows(result.time_stamps)
    cues, mapping_diag, mapped_rows = build_subtitle_cues(
        result.text, rows,
        max_seconds=SUBTITLE_MAX_SECONDS,
        max_chars=SUBTITLE_MAX_CHARS,
        gap_break_seconds=SUBTITLE_GAP_BREAK_SECONDS,
        min_sentence_break_seconds=SUBTITLE_MIN_SENTENCE_BREAK_SECONDS,
        max_line_chars=SUBTITLE_MAX_LINE_CHARS,
    ) if rows else ([], {
        "punctuation_mapping_exact": True,
        "alignment_mapping_accepted": True,
        "alignment_similarity": 1.0,
        "alignment_normalized_exact": True,
        "alignment_unmatched_source_chars": 0,
        "alignment_unmatched_aligner_chars": 0,
        "timestamp_inversions_before": 0,
        "timestamp_inversions_after": 0,
        "timestamp_repair_count": 0,
        "timestamp_repairs": [],
        "timestamp_mapping_monotonic": True,
    }, [])
    payload = {
        "source_file": Path(src).name,
        "model": model_id,
        "language": result.language,
        "text": result.text,
        "timestamp_mode": TIMESTAMP_MODE,
        "punctuation_mapping_exact": mapping_diag.get("punctuation_mapping_exact"),
        "alignment_mapping": mapping_diag,
        "alignment": mapped_rows if mapped_rows else rows,
        "alignment_raw": rows,
        "subtitle_cues": cues,
    }

    saved = []
    if SAVE_TXT:
        p = out_dir / f"{base}.txt"
        if INCLUDE_TIMESTAMPS_IN_TXT and cues:
            body = "\n".join(f"[{fmt_ts(c['start'])} --> {fmt_ts(c['end'])}] {c['text']}" for c in cues)
        else:
            body = result.text
        p.write_text(body + "\n", encoding="utf-8")
        saved.append(p)

    if SAVE_MD:
        p = out_dir / f"{base}.md"
        parts = [
            f"# {Path(src).name}",
            "",
            f"- Model: `{model_id}`",
            f"- Language: `{result.language}`",
            f"- Timestamp mode: `{TIMESTAMP_MODE}`",
            "",
            "## Transcript",
            "",
            result.text,
        ]
        if cues:
            parts += ["", "## Timestamped transcript", ""]
            parts += [f"- `{fmt_ts(c['start'])}–{fmt_ts(c['end'])}` {c['text']}" for c in cues]
        p.write_text("\n".join(parts) + "\n", encoding="utf-8")
        saved.append(p)

    if SAVE_JSON:
        p = out_dir / f"{base}.json"
        p.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
        saved.append(p)

    table_rows = mapped_rows if mapped_rows else (rows if rows else [{"text": result.text, "start_time": None, "end_time": None}])
    if SAVE_CSV:
        p = out_dir / f"{base}.csv"
        pd.DataFrame(table_rows).to_csv(p, index=False, encoding="utf-8-sig")
        saved.append(p)
    if SAVE_XLSX:
        p = out_dir / f"{base}.xlsx"
        with pd.ExcelWriter(p, engine="openpyxl") as writer:
            pd.DataFrame(table_rows).to_excel(writer, index=False, sheet_name="alignment")
            if cues:
                pd.DataFrame(cues).to_excel(writer, index=False, sheet_name="subtitles")
        saved.append(p)
    if SAVE_SRT and cues:
        p = out_dir / f"{base}.srt"
        p.write_text(cues_to_srt(cues), encoding="utf-8")
        saved.append(p)
    return saved, payload


In [ ]:
# @title 5. Qwen3-ASRをロード
# ============================================================
# 5. Qwen3-ASRをロード
# ============================================================
import gc, torch
from qwen_asr import Qwen3ASRModel

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
forced_aligner = ALIGNER_ID if RETURN_TIMESTAMPS else None
forced_aligner_kwargs = (
    {"dtype": dtype, "device_map": "cuda:0"} if RETURN_TIMESTAMPS else None
)

print("読み込み中:", MODEL_ID)
print("dtype:", dtype)
print("forced_aligner:", forced_aligner)

model = Qwen3ASRModel.from_pretrained(
    MODEL_ID,
    dtype=dtype,
    device_map="cuda:0",
    max_inference_batch_size=MAX_INFERENCE_BATCH_SIZE,
    max_new_tokens=MAX_NEW_TOKENS,
    forced_aligner=forced_aligner,
    forced_aligner_kwargs=forced_aligner_kwargs,
)

print("モデルを読み込みました。")


In [ ]:
# @title 6. 文字起こし
# ============================================================
# 6. 文字起こし
# ============================================================
import gc, json, shutil, traceback
from pathlib import Path

language_arg = None if SOURCE_LANGUAGE == "Auto" else SOURCE_LANGUAGE
all_saved = []
manifest = []

for index, src in enumerate(input_files, 1):
    print(f"\n[{index}/{len(input_files)}] {src}")
    try:
        wav = extract_audio(src)
        results = model.transcribe(
            audio=str(wav),
            context=CONTEXT_INFO,
            language=language_arg,
            return_time_stamps=RETURN_TIMESTAMPS,
        )
        result = results[0]
        saved, payload = save_result(src, MODEL_ID, result)
        all_saved.extend(saved)
        manifest.append({
            "source": Path(src).name,
            "status": "ok",
            "language": result.language,
            "characters": len(result.text),
            "alignment_items": len(payload["alignment"]),
            "subtitle_cues": len(payload["subtitle_cues"]),
            "punctuation_mapping_exact": payload["punctuation_mapping_exact"],
            "cue_text_exact": payload.get("alignment_mapping", {}).get("cue_text_exact"),
            "timestamp_mapping_monotonic": payload.get("alignment_mapping", {}).get("timestamp_mapping_monotonic"),
            "timestamp_repair_count": payload.get("alignment_mapping", {}).get("timestamp_repair_count"),
            "outputs": [p.name for p in saved],
        })
        print("検出言語:", result.language)
        print("文字数:", len(result.text))
        print("アラインメント項目数:", len(payload["alignment"]))
        print("字幕cue数:", len(payload["subtitle_cues"]))
        print("句読点マッピング完全一致:", payload["punctuation_mapping_exact"])
    except torch.cuda.OutOfMemoryError:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        manifest.append({"source": Path(src).name, "status": "cuda_oom"})
        print("CUDA OOM. MODEL_SIZE=0.6B を選ぶか、タイムスタンプを none にして再実行してください。")
        raise
    except Exception as e:
        manifest.append({"source": Path(src).name, "status": "error", "error": repr(e)})
        traceback.print_exc()
        continue
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

manifest_path = Path(OUTPUT_DIR) / "manifest.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
all_saved.append(manifest_path)

print("\n処理が完了しました。")
for p in all_saved:
    print("-", p)


In [ ]:
# @title 7. ZIP保存・ダウンロード
# ============================================================
# 7. ZIP保存・ダウンロード
# ============================================================
from pathlib import Path
import shutil, zipfile

zip_path = Path("/content") / ZIP_FILE_NAME
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(Path(OUTPUT_DIR).glob("*")):
        if p.is_file():
            zf.write(p, arcname=p.name)

print("ZIP:", zip_path)

if SAVE_OUTPUTS_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    drive_out = Path(GOOGLE_DRIVE_ROOT) / "Qwen3_ASR_outputs"
    drive_out.mkdir(parents=True, exist_ok=True)
    shutil.copy2(zip_path, drive_out / zip_path.name)
    print("Google Driveに保存しました:", drive_out / zip_path.name)

if AUTO_DOWNLOAD_ZIP:
    from google.colab import files
    files.download(str(zip_path))
